<a href="https://colab.research.google.com/github/Barna036/Intro-to-Python1/blob/main/Day_5_Scraping_Practice_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5: Web scraping and text-as-data

The plan:

- **APIs vs scraping** — when to use which
- **BeautifulSoup basics** on a small HTML document
- **Polite scraping**: `robots.txt` + sleep between requests
- **A real scrape**: the 2024 Democratic Party Platform
- **GroupAppeals**: running a transformer model on the scraped text

## Setup

In [ ]:
import requests
import pandas as pd
import re
import time

# Custom user-agent — many sites reject the default python-requests/X.Y.Z
HEADERS = {
    "User-Agent": "ICPSR-Python-Course/1.0 (rwhorne@clemson.edu)"
}

## APIs vs scraping

If you can get what you need from an API, **use the API**. They return structured data, are well-maintained, and don't break when the website redesigns. You scrape when:

- The API doesn't exist
- The API has rate limits or costs money (and you aren't at, say, Harvard)
- The API doesn't expose the data you actually need

If a webpage is served to your browser, you can ask for it from Python too. That's all scraping is.

## BeautifulSoup: the basics

We'll start with a toy HTML snippet so you can see the navigation patterns without network calls in the loop.

In [ ]:
html_doc = """
<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>
"""

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_doc, "html.parser")
print(soup.prettify())

<html>
 <head>
  <title>
   The Dormouse's story
  </title>
 </head>
 <body>
  <p class="title">
   <b>
    The Dormouse's story
   </b>
  </p>
  <p class="story">
   Once upon a time there were three little sisters; and their names were
   <a class="sister" href="http://example.com/elsie" id="link1">
    Elsie
   </a>
   ,
   <a class="sister" href="http://example.com/lacie" id="link2">
    Lacie
   </a>
   and
   <a class="sister" href="http://example.com/tillie" id="link3">
    Tillie
   </a>
   ;
and they lived at the bottom of a well.
  </p>
  <p class="story">
   ...
  </p>
 </body>
</html>



`BeautifulSoup` parses HTML into a tree you can navigate by tag name or attribute.

### Navigating the structure

In [ ]:
soup.title              # the whole <title> tag

In [ ]:
soup.title.string       # just the text inside

In [ ]:
soup.title.parent.name  # what tag contains <title>?

When the same tag appears multiple times, `soup.tag` gives you the **first** one only:

In [ ]:
soup.p   # first <p>

To get them all, use `find_all`:

In [ ]:
soup.find_all("p")   # every <p>

### Finding by id

When elements have `id` attributes, you can grab them directly:

In [ ]:
soup.find(id="link3")

### Extracting all the links

In [ ]:
for link in soup.find_all("a"):
    print(link.get("href"))

### Extracting just the text

In [2]:
print(soup.get_text())

NameError: name 'soup' is not defined

### Your Turn — work with the soup

Using the same `soup` object:

1. Print the text content of just the first paragraph (the title "The Dormouse's story")
2. Find all `<a>` tags with `class="sister"` and print just their text — should be `Elsie`, `Lacie`, `Tillie`
3. Use `soup.find(id="link2")` to get the Lacie link. What does `.get("href")` return on it?

class_="sister" (with underscore!) because class is a reserved Python keyword, so BeautifulSoup uses class_ instead for the keyword argument

## Browser dev tools

The trick to scraping any real page is figuring out *which selector* gives you the part you want. Browsers have built-in tools for this.

In Chrome/Firefox: right-click an element on the page and choose **Inspect**. The dev tools open with the corresponding HTML highlighted. You can read off the tag, class, and id you need to target.

We'll use this in a few minutes when we scrape a real document.

##if not work, shoot email them, or use \sitempa ## robots.txt — the polite scraping checklist

`robots.txt` is where a site tells bots what they can and can't access. It lives at `<site>/robots.txt`. Example:

```
User-agent: *
Disallow: /admin/
Disallow: /api/private/
```

Why bother?

- It's good etiquette
- IP bans are common for sites you ignore
- Repeated violations can have legal consequences (especially commercial scraping)

If `robots.txt` doesn't list what you need, check `/sitemap.xml`.

## Don't crash the site you're scraping

If you're scraping many pages from the same site, **add a delay** between requests. Even a couple of seconds is enough to avoid being rate-limited or banned.

In [1]:
## example.com/page1 doesn't exist — these will 404, but it demos the sleep pattern.
urls = [
    "https://example.com/page1",
    "https://example.com/page2",
    "https://example.com/page3",
]

for url in urls:
    response = requests.get(url, headers=HEADERS)
    if response.status_code < 300:
        print(f"  Success! Got {len(response.text)} bytes from {url}")
    else:
        print(f"  Failed ({response.status_code}) on {url}")

    # Be polite
    time.sleep(2)

NameError: name 'requests' is not defined

### Your Turn — pull section headings from a Wikipedia article

1. Use `requests.get()` with `headers=HEADERS` to fetch
   <https://en.wikipedia.org/wiki/Web_scraping>
2. Parse the response into `wiki_soup`
3. Wikipedia's section headings are inside `<h2>` tags. Use
   `wiki_soup.find_all("h2")` and print just the text of each one.
4. Find the first paragraph of the article (`<p>` tag) and print its
   text.

In [38]:
r = requests.get(url= "https://en.wikipedia.org/wiki/Web_scraping", headers=HEADERS)
# wiki_soup = BeautifulSoup(r.text, "html.parser")

# url = "https://en.wikipedia.org/wiki/Web_scraping"
# HEADERS = {"User-Agent": "barna036@gmail.com"}

# response = requests.get(url, headers=HEADERS)

# wiki_soup = BeautifulSoup(response.text, "html.parser")

# for heading in wiki_soup.find_all("h2"):
#     print(heading.get_text())

# first_paragraph = wiki_soup.find("p")
# print(first_paragraph.get_text())

In [37]:
wiki_soup = BeautifulSoup(r.text, "html.parser")

for heading in wiki_soup.find_all("h2"):
    print(heading.get_text())

Documents
Categories
Democratic National ConventionLand Acknowledgement
Preamble
Table of Contents
Chapter One: Growing Our Economy from the Bottom Up & Middle Out
Chapter Two: Rewarding Work, Not Wealth
Chapter Three: Lowering Costs
Chapter Four: Tackling the Climate Crisis, Lowering Energy Costs, & Securing Energy Independence
Chapter Five: Protecting Communities & Tackling the Scourge of Gun Violence
Chapter Six: Strengthening Democracy, Protecting Freedoms, & Advancing Equity
Chapter Seven: Securing our Border & Fixing the Broken Immigration System
Chapter Eight: Advancing the President's Unity Agenda
Chapter Nine: Strengthening American Leadership Worldwide
Related PDFs
Simple Search of Our Archives


## A real scrape: the 2024 Democratic Party Platform

We're going to scrape an actual political document — the 2024 Democratic Party Platform — hosted at the [American Presidency Project](https://www.presidency.ucsb.edu/) at UCSB.

Why this website?

- Static HTML, no JavaScript rendering surprises
- The document body sits in a single identifiable container
- Stable URL, it won't disappear next semester
- The American Presidency Project is widely cited in political science research
- I kind of want to see how the models I trained work on American data!

In [39]:
URL = "https://www.presidency.ucsb.edu/documents/2024-democratic-party-platform"

r = requests.get(URL, headers=HEADERS)
print("Status:", r.status_code)   # should be 200

Status: 200


In [36]:
soup = BeautifulSoup(r.text, "html.parser")
soup.title.string

'2024 Democratic Party Platform | The American Presidency Project'

### Extracting the document body

If you inspect the page in a browser, the platform text sits inside `<div class="field-docs-content">`. A single CSS selector grabs everything:

In [41]:
content = soup.select_one("div.field-docs-content")
text = content.get_text(separator=" ", strip=True)

print(text[:300])
print()
print(len(text), "characters total")

Democratic National Convention Land Acknowledgement The Democratic National Committee wishes to acknowledge that we gather together to state our values on lands that have been stewarded through many centuries by the ancestors and descendants of Tribal Nations who have been here since time immemorial

281916 characters total


`select_one(...)` is BeautifulSoup's CSS-selector shortcut. `"div.field-docs-content"` means "a `<div>` whose class includes `field-docs-content`." The dot prefix is the CSS class syntax — same as you'd use in a stylesheet.

## Cleaning and formatting for analysis

For text analysis we usually want one sentence per row, with metadata for traceability:

In [48]:
# Split on sentence-ending punctuation. Quick-and-dirty, but works.
#Espace e pacakge for better
sentences = r.split(r"(?<=[.!?])\s+", text)

# Drop empties and very short fragments
sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

df = pd.DataFrame({
    "party":       "Democratic",
    "date":        "2024",
    "sentence_id": range(len(sentences)),
    "text":        sentences,
})

print(f"{len(df)} sentences")
df.head()

AttributeError: 'Response' object has no attribute 'split'

The regex `(?<=[.!?])\s+` says "split on whitespace that's preceded by `.`, `!`, or `?`." Quick approach — for production work, use spaCy's sentence segmenter (it handles abbreviations like `Dr.` and `etc.` properly).

### Your Turn — explore the scraped text

1. How many sentences in `df` are longer than 100 characters? (Hint: `df["text"].str.len() > 100`)
2. What's the longest sentence? Print it.
3. How many sentences contain the word "workers"? (Hint: `.str.contains("workers", case=False)`)

## GroupAppeals: a transformer model on the scraped text

`GroupAppeals` (full disclosure: I'm a co-author) analyzes how political parties reference and appeal to social groups. It runs four fine-tuned transformer models:

- **Token extraction** — identifies group references (`workers`, `immigrants`, `families`)
- **Stance detection** — positive, negative, or neutral toward the group
- **Policy detection** — does the text propose a policy aimed at the group?
- **Group classification** — maps the reference to a semantic category

First run installs ~1.5GB of weights — be patient.

In [44]:
!pip install --quiet groupappeals

In [45]:
from groupappeals.fullpipeline import run_full_pipeline

### Running the pipeline

For a live demo we take only ~30 sentences so the run finishes in a couple of minutes on Colab CPU. For the full platform you'd want a GPU runtime.

In [46]:
## The platform begins with a land acknowledgement, which doesn't have much in the
## way of policies. The model actually did OK with the tribes (I checked!), but let's look
## at something in the heart of the document

sample = df.iloc[18:51].copy()
sample.to_csv("dnc_sample.csv", index=False)
sample.head()




NameError: name 'df' is not defined

#check huggingface for modeling

birdmodels,

In [ ]:
results = run_full_pipeline(
    input_file="dnc_sample.csv",
    output_file="dnc_results.csv",
    create_composite_id=["party", "date", "sentence_id"],
)

print(f"Analyzed {len(results)} rows")
results.head()

🚀 Starting Full GroupAppeals Analysis Pipeline

🔧 Step 0: Creating composite IDs from columns: ['party', 'date', 'sentence_id']
✅ Step 0 Complete: Created 33 composite IDs

🔍 Step 1: Extracting group references with token classification...
Using device: cpu


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Classifying: 100%|██████████| 33/33 [00:12<00:00,  2.55it/s]


Results saved to: ./temp_step1_dnc_results.csv
✅ Step 1 Complete: Extracted groups from 47 rows

🔄 Step 2: Filtering data for stance and policy detection...
✅ Step 2 Complete: Filtered 29 rows for stance/policy detection
   (Separated 18 rows with missing group references)

😊 Step 3: Detecting stance towards groups...
Using device: cpu


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Extracting clean policy labels...
Results saved to: ./temp_step4_dnc_results.csv
   🧹 Applying policy label post-processing...
   Clean policy distribution:
     no policy: 19
     policy: 10
✅ Step 4 Complete: Policy detection finished

🏷️  Step 5: Classifying groups into meaningful categories...
Using device: cpu


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Results saved to: ./temp_step5_dnc_results.csv
   🧹 Applying meaningful groups post-processing...
   Category distribution:
     Citizens: 7
     Families: 4
     Other: 4
     Upper Class: 3
     Military Personnel: 3
     Lower Class: 2
     Criminals: 2
     Women: 2
     Law Enforcement Personnel: 1
     Students: 1
     Politicians: 1
     Young People: 1
     Children: 1
   🔄 Splitting meaningful groups into separate columns...
Auto-determined max groups: 2
Created 2 group columns: ['Group1', 'Group2']
✅ Step 5 Complete: Group classification finished

🔗 Step 6: Preparing final dataset...
✅ Row count validation passed: 47 total rows
✅ Coverage validation passed: all 33 unique base text_ids preserved
✅ Step 6 Complete: Combined full analysis and no-groups data
   Final dataset contains 47 rows with 15 columns
   - Rows with full analysis: 29
   - Rows without group detection: 18

💾 Saving final results to: dnc_results.csv
✅ Results successfully saved to: dnc_results.csv

🧹 Cleaning

/usr/local/lib/python3.12/dist-packages/groupappeals/fullpipeline.py:410: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat([full_models_output, no_detected_groups], ignore_index=True)


,text_id,text,Exact.Group.Text,Average Score,Start,End,Stance,Stance_Confidence,Stance_Clean,Policy,Policy_Confidence,Policy_Clean,Meaningful Group,Group1,Group2
0,Democratic_2024_19.1,We lowered families' health insurance premiums...,families,0.999165,10.0,19.0,The text is positive towards families.,0.999965,positive,The text contains a policy directed towards f...,0.999728,policy,[Families],Families,None
1,Democratic_2024_20.1,We're helping students catch up on learning lo...,students,0.997750,13.0,22.0,The text is positive towards students .,0.999954,positive,The text does not contain a policy directed to...,0.998663,no policy,[Students],Students,None
2,Democratic_2024_21.1,We passed the first significant gun safety law...,police officers,0.999398,116.0,132.0,The text is positive towards police officers .,0.999975,positive,The text contains a policy directed towards p...,0.999863,policy,[Law Enforcement Personnel],Law Enforcement Personnel,None
3,Democratic_2024_26.1,We've expanded benefits and services for toxic...,toxic-exposed veterans,0.999744,40.0,63.0,The text is positive towards toxic-exposed ve...,0.999986,positive,The text contains a policy directed towards t...,0.999894,policy,[Military Personnel],Military Personnel,None
4,Democratic_2024_26.2,We've expanded benefits and services for toxic...,American,0.979886,107.0,116.0,The text is positive towards American .,0.525503,positive,The text contains a policy directed towards A...,0.997519,policy,[Citizens],Citizens,None


Note - we could run this on the GPUs that come with an academic or pro account on colab - but I'd imagine that not all of you have activated that yet. That would make this run a little faster, but if we ran the whole data set in would be at least 5x faster.

### Exploring the results

In [ ]:
results[[
    "text_id",
    "Exact.Group.Text",   # the extracted group reference
    "Stance_Clean",       # positive / negative / neutral
    "Policy_Clean",       # policy / no policy
    "Group1",             # semantic category
]].head(10)

,text_id,Exact.Group.Text,Stance_Clean,Policy_Clean,Group1
0,Democratic_2024_19.1,families,positive,policy,Families
1,Democratic_2024_20.1,students,positive,no policy,Students
2,Democratic_2024_21.1,police officers,positive,policy,Law Enforcement Personnel
3,Democratic_2024_26.1,toxic-exposed veterans,positive,policy,Military Personnel
4,Democratic_2024_26.2,American,positive,policy,Citizens
5,Democratic_2024_28.1,everyone,positive,policy,Citizens
6,Democratic_2024_30.1,families,positive,no policy,Families
7,Democratic_2024_32.1,the American people,negative,no policy,Citizens
8,Democratic_2024_33.1,extreme MAGA allies,negative,no policy,Criminals
9,Democratic_2024_33.2,women,positive,policy,Women


## Wrap-up

You scraped a real political document and ran it through a fine-tuned transformer classifier — that's a lot of ground for one day.

Tools worth exploring next:
- **spaCy** — industrial-strength NLP pipeline
- **scikit-learn** — classic ML for text classification (TF-IDF + logistic regression is a strong baseline)
- **HuggingFace Transformers** — state-of-the-art language models
- **NLTK** — older but still useful for classic NLP tasks
- **The [Manifesto Project](https://manifesto-project.wzb.eu/)** — coded political party manifestos across 50+ countries. Great to practice on.
- Or, **try our data here**: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/FVI3QW


Reach out if you have questions later. Good luck with the rest of your research.